# NOD intrusion sweep

How much does the headline NOD incline (Late.NOD.Wake - Early.NOD.Wake) depend on brief
non-Wake bouts that were scored as sleep during the deprivation window? Those bouts are
candidate intrusions: mislabeled local sleep whose OFF periods the canonical Wake
buckets exclude.

For each duration threshold `T`, the sweep admits intrusion bouts of duration <= `T`,
adds their OFFs to the fixed Early/Late.NOD.Wake buckets (denominators unchanged), and
recomputes the chosen metric. `T = 0` admits nothing and reproduces the canonical
result.

This notebook drives
[`cnpix_local_sleep.morphological.mua.pipeline.full48h`](../../src/cnpix_local_sleep/morphological/mua/pipeline/full48h.py)
for any additive OFF metric (count, rate, total area). The sweep is only valid for
metrics that grow by addition. It is `morphological`-only and reads the full-48h
`offs.parquet` aggregates, so it requires mounted production data (`/Volumes/npx_nfs/`).
Distributional metrics (median span, mean duration, and so on) are not additive and are
out of scope.

Outputs are publication SVGs, via the `pubplots` figma destination, under
`outputs/intrusion_sweep/<metric>/`. Per measure there are two 3-row stacked figures,
one row per OFF set, ~1.5 inches wide by 3.3 tall, built to line up row-for-row with the
`incline_magnitudes` figures:

- `inclusive.svg`: BLAS (top), CLAS, LLAS (bottom)
- `exclusive.svg`: BLAS (top), CLAS-exclusive, LLAS-exclusive (bottom)

plus per-combo 3x3 detail figures under `by_structure/`. The exclusive categories are
the adjacent partition (`CLAS-exclusive = CLAS\BLAS`, `LLAS-exclusive = LLAS\CLAS`), so
BLAS + CLAS-exclusive + LLAS-exclusive reconstructs LLAS.


In [ ]:
from IPython.display import SVG

from cnpix_local_sleep import sps_conf
from cnpix_local_sleep.morphological.mua.pipeline import full48h

## Configuration

Pick the measures to sweep. `ADDITIVE_METRICS` lists everything the sweep
supports; `INTRUSION_STATES` are the non-Wake states whose NOD bouts count as
candidate intrusions. The OFF sets are fixed by the two stacked figures
(`INCLUSIVE_STACK` and `EXCLUSIVE_STACK`).

In [ ]:
METRICS = ("count", "total_area_norm")

print("supported additive metrics:", sorted(full48h.ADDITIVE_METRICS))
print("intrusion states:", full48h.INTRUSION_STATES)
print("inclusive stack (top->bottom):", full48h.INCLUSIVE_STACK)
print("exclusive stack (top->bottom):", full48h.EXCLUSIVE_STACK)
print("output root:", full48h.INTRUSION_OUTPUT_DIR)

## Sanity check: extraction coverage

Confirm the full-48h `offs.parquet` exists for the analysis combos and
carries a per-OFF `state` column (needed to find intrusion bouts).

In [ ]:
coverage = full48h.verify_extraction()
coverage

## Single-combo inspection

Run the sweep for one (subject, probe, structure) to see the two tables it
returns: the per-threshold `sweep` and the per-bout `intrusions`. We pick the
first analysis combo, dropping the subjects whose `Early.NOD.Wake` window is
unreliable (their incline is undefined).

In [ ]:
spsl = [
    s
    for s in sps_conf.get_analysis_spsl()
    if s[0] not in full48h.NOD_INCLINE_EXCLUDED_SUBJECTS
]
subject, probe, structure = spsl[0]

sweep, intrusions = full48h.intrusion_sweep(
    subject, probe, structure, metric="total_area_norm", filter_name="llas"
)
print(f"{subject} {probe} {structure}")
sweep

In [ ]:
# One row per intrusion bout: state, duration, early/late/middle bucket, and
# its contribution to the (un-normalized) metric numerator.
intrusions.sort_values("duration").head(10)

Render the three-panel figure for this combo and show it inline. The same
call is what the batch driver uses; it writes the SVG under
`intrusion_sweep/<metric>/<filter>/by_structure/`.

In [ ]:
svg_path = full48h.plot_intrusion_sweep(subject, probe, structure, sweep)
print(svg_path)
SVG(filename=str(svg_path))

## Batch: compute (cached) + plot

The slow step (`compute_intrusion_sweeps`, a few minutes: one full-48h `offs.parquet`
load per combo x filter x measure) is separated from plotting. The first cell loads the
cached sweep table if present, else computes and caches it to `sweep_results.parquet`.
The second cell renders all figures from that table in seconds, so after changing a plot
parameter re-run only the plot cell, with no recompute. Set `REFRESH=True` after
changing `METRICS`, the combo list, or the thresholds.


In [ ]:
# Compute (or load cached) the sweep table. This is the ~minutes step; the cache
# lets the plot cell below re-render in seconds. Set REFRESH=True after changing
# METRICS, the combo list, or thresholds (the cache is keyed only by path).
REFRESH = False
CACHE_PATH = full48h.INTRUSION_OUTPUT_DIR / "sweep_results.parquet"

results = full48h.load_or_compute_intrusion_sweeps(
    CACHE_PATH, refresh=REFRESH, metrics=METRICS
)
print("cache:", CACHE_PATH)
results.groupby(["filter", "metric"]).size().rename("n_rows")

In [ ]:
# Plot from the (cached) table: fast, no NFS reads. Re-run just this cell to
# refresh the figures after changing a plot parameter.
#
# LOG_Y symlog-scales the y axis; it tames the one wildly-scaled combo but gives
# the sub-zero excursions their own decades. On a linear axis, prefer capping the
# affected row instead: YLIMS is {metric: {off_set: (low, high)}}, either bound
# None to autoscale. Here BLAS count is capped at 100 (clips the outlier, keeps
# the near-zero / negative region honest); other rows autoscale.
LOG_Y = False
YLIMS = {
    "count": {"blas": (None, 125)},
}

full48h.plot_intrusion_sweeps(
    results,
    plot_for_pub=True,
    log_y=LOG_Y,
    ylims=YLIMS,
    by_structure=True,
)

Where the figures landed:

In [ ]:
root = full48h.INTRUSION_OUTPUT_DIR
stacks = sorted(root.glob("*/inclusive.svg")) + sorted(root.glob("*/exclusive.svg"))
per_combo = sorted(root.glob("*/by_structure/*.svg"))
print(f"{len(stacks)} stacked figures, {len(per_combo)} per-structure detail figures")
for p in sorted(stacks):
    print(" ", p.relative_to(root))

Compare the incline across filters/metrics at the extremes of the sweep,
`T = 0` (canonical) vs the widest admitted threshold, to read off how much
admitting intrusions moves each contrast.

In [ ]:
tmax = results["threshold_s"].max()
ends = results[results["threshold_s"].isin([0, tmax])]
table = ends.pivot_table(
    index=["filter", "metric"],
    columns="threshold_s",
    values="incline",
    aggfunc="mean",
)
table.columns = [f"incline@T={int(c)}" for c in table.columns]
table["delta"] = table[f"incline@T={int(tmax)}"] - table["incline@T=0"]
table